In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedGroupKFold

SEED = 42

df = pd.read_parquet("data/solubility_clustered.parquet")
pool = df[df["stage"] != "test"].copy()

In [3]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_pos, test_pos = next(sgkf.split(pool, pool["label"], groups=pool["cluster"]))

clu_train = pool.index[train_pos]
clu_test = pool.index[test_pos]

In [4]:
print(len(clu_test))
print(pool.loc[clu_test, "label"].mean())

13885
0.4173568599207778


In [5]:
rand_train, rand_test = train_test_split(
    pool.index,
    test_size=len(clu_test),
    stratify=pool["label"],
    random_state=SEED,
)

In [6]:
print(pool.loc[clu_test, "label"].mean())
print(pool.loc[rand_test, "label"].mean())

0.4173568599207778
0.4173568599207778


In [7]:
def shared_clusters(train_idx, test_idx):
    train_c = set(pool.loc[train_idx, "cluster"])
    test_c = set(pool.loc[test_idx, "cluster"])
    return len(train_c & test_c)

print(shared_clusters(clu_train, clu_test))
print(shared_clusters(rand_train, rand_test))

0
5023


In [8]:
rand_train_clusters = set(pool.loc[rand_train, "cluster"])
pool.loc[rand_test, "cluster"].isin(rand_train_clusters).mean()

np.float64(0.5748649621894131)

In [9]:
pool["split_random"] = "train"
pool.loc[rand_test, "split_random"] = "test"

pool["split_cluster"] = "train"
pool.loc[clu_test, "split_cluster"] = "test"

In [10]:
pool.to_parquet("data/pool_splits.parquet")